<a href="https://colab.research.google.com/github/fkyria/vlm-robotics-finetune/blob/main/finetune_vlm_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Fine-tune Qwen2-VL-2B with QLoRA on Robo2VLM-1 (robotics VQA)

Runs on a free Colab T4 GPU, connected from VS Code via the Google Colab extension.

### 1. Install dependencies

In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 25.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.9 MB/s eta 0:00:00:00:0100:01


### 2. Imports and Configuration

In [2]:
import ast
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_DIR = "qwen2vl-2b-robo2vlm-lora"


### 3. Load the model in 4-bit (QLoRA)

Weights are stored in 4-bit and de-quantized on the fly for compute; only the tiny
LoRA adapters (added in the next cell) are trained in full precision.

- `load_in_4bit=True` —> weights stored at 4 bits instead of 16 (~4x less memory).
- `bnb_4bit_quant_type="nf4"` —> the specific 4-bit encoding scheme, tuned for how
  neural network weights are actually distributed (clustered near zero).
- `bnb_4bit_use_double_quant=True` —>  also quantizes the small scaling factors
  quantization itself needs, for a bit more memory savings.
- `bnb_4bit_compute_dtype=torch.bfloat16` —>  math is temporarily done in bfloat16
  even though storage is 4-bit.


In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

### 4. Attach a LoRA adapter

Freezes the base model; injects small trainable low-rank matrices into attention
(`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) layers.

- `r=16` —>  adapter rank. Higher = more capacity, more memory. 8–32 is typical for small tasks.
- `lora_alpha=32` —>  scales the adapter's influence; kept at the standard `2×r` ratio.
- `lora_dropout=0.05` —>  randomly drops 5% of adapter connections per step, to avoid overfitting.

Run `print_trainable_parameters()` —>  it should show trainable params at well under 1%
of the full model.


In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


### 5. Load and format the dataset

[Robo2VLM-1](https://huggingface.co/datasets/keplerccc/Robo2VLM-1): multiple-choice VQA
grounded in real robot manipulation footage.        
Schema: `question`, `choices` (stringified list), `correct_answer` (int index), `image`.

`to_chat_format` converts one raw row into the chat-message structure the processor's
chat template expects:      
image + question + options as the user turn, the correct letter as the assistant turn.


In [5]:
N_TRAIN = 200
N_VAL = 20
N_TEST = 100

raw_stream = load_dataset("keplerccc/Robo2VLM-1", split="train", streaming=True)
raw_examples = list(raw_stream.take(N_TRAIN + N_VAL + N_TEST))

train_raw = raw_examples[:N_TRAIN]
val_raw = raw_examples[N_TRAIN : N_TRAIN + N_VAL]
test_raw = raw_examples[N_TRAIN + N_VAL:]

SYSTEM_MESSAGE = (
    "You are a helpful assistant that answers multiple-choice questions "
    "about a robot's manipulation task shown in the image. Respond with "
    "only the letter of the correct choice."
)

def to_chat_format(example):
    choices = ast.literal_eval(example["choices"])  # "[\'a\',\'b\']" -> [\'a\', \'b\']
    letters = [chr(ord("A") + i) for i in range(len(choices))]
    options_text = "\n".join(f"{letters[i]}. {choices[i]}" for i in range(len(choices)))
    answer_letter = letters[example["correct_answer"]]

    return {
        "images": [example["image"]],
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": f"{example['question']}\n{options_text}"},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": answer_letter},
            ]},
        ],
    }


train_dataset = Dataset.from_list([to_chat_format(ex) for ex in train_raw])
val_dataset = Dataset.from_list([to_chat_format(ex) for ex in val_raw])


README.md:   0%|          | 0.00/2.73k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/262 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/262 [00:00<?, ?it/s]

### 6. Collate function
Turns a batch of chat-formatted examples into model-ready tensors:
- `processor(...)` tokenizes text and converts images into normalized pixel tensors, in one call.
- `labels` starts as a copy of `input_ids` (next-token prediction), then padding tokens
  and image placeholder tokens are set to `-100` — the sentinel value that tells the loss
  function "ignore this position." Without this the model wastes effort trying to
  predict padding and image tokens instead of just the answer.

In [6]:
def collate_fn(examples):
    texts = [
        processor.apply_chat_template(ex["messages"], tokenize=False)
        for ex in examples
    ]
    images = [ex["images"] for ex in examples]

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    image_token_id = processor.tokenizer.convert_tokens_to_ids(
        processor.image_token if hasattr(processor, "image_token") else "<|image_pad|>"
    )
    if image_token_id is not None:
        labels[labels == image_token_id] = -100

    batch["labels"] = labels
    return batch

### 7. Train

- `per_device_train_batch_size=1`, `gradient_accumulation_steps=8` —>  effective batch
  size 8, built by accumulating gradients across 8 single-example steps (images make
  batch size 1 the practical ceiling on a T4).
- `learning_rate=2e-4` —>  higher than full fine-tuning uses, because these are fresh,
  small LoRA params, not an already-trained giant weight matrix.
- `bf16=True` —>  training math in bfloat16, big memory savings for negligible accuracy loss.
- `gradient_checkpointing=True` —>  recomputes activations during backward pass instead
  of storing them, trading compute time for memory.
- `remove_unused_columns=False` —>  keeps the `images` column alive so `collate_fn` can see it.


In [7]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop if eval_loss stalls for 3 evals
)

trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
5,2.449074,1.438698,1.460014,15073.000000,0.750935
10,1.145132,0.902094,0.906198,30161.000000,0.828036
15,0.696384,0.651199,0.669183,45002.000000,0.869760
20,0.494185,0.480692,0.503676,60053.000000,0.904749
25,0.383160,0.417921,0.428339,74828.000000,0.918638
30,0.350284,0.374897,0.357758,89998.000000,0.929167
35,0.315252,0.374259,0.346130,105053.000000,0.932124
40,0.347837,0.364637,0.348944,120025.000000,0.933806
45,0.283603,0.355462,0.326062,135144.000000,0.935914
50,0.273638,0.351611,0.314921,149656.000000,0.935928


TrainOutput(global_step=90, training_loss=0.49112263653013444, metrics={'train_runtime': 3890.5882, 'train_samples_per_second': 0.206, 'train_steps_per_second': 0.026, 'total_flos': 3219468692170752.0, 'train_loss': 0.49112263653013444, 'epoch': 3.6})

### 8. Save the adapter
Only the small LoRA weights (a few MB), not the full 2B-parameter model.

In [8]:
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
processor.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

Adapter saved to qwen2vl-2b-robo2vlm-lora/final_adapter


### 9. Validation accuracy.
What does it actually get right?

Runs the fine-tuned model on `val_raw` (20 examples, never trained on directly, but used
during training to pick the best checkpoint) and checks whether its generated answer
matches the correct letter. Unlike the loss numbers above, this uses real generation
(`model.generate()`). The model produces its own answer from scratch, rather than being
shown the correct next token and graded on that.

- Used as sanity check but not fully unbiased as `val_raw` influenced checkpoint selection.


In [10]:
model.eval()

correct = 0
for example in val_raw:
    choices = ast.literal_eval(example["choices"])
    letters = [chr(ord("A") + i) for i in range(len(choices))]
    options_text = "\n".join(f"{letters[i]}. {choices[i]}" for i in range(len(choices)))
    true_letter = letters[example["correct_answer"]]

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": f"{example['question']}\n{options_text}"},
        ]},
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[[example["image"]]], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    predicted_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    predicted_letter = predicted_text[:1].upper()

    is_correct = predicted_letter == true_letter
    correct += is_correct
    print(f"{"correct" if is_correct else "incorrect"} predicted={predicted_letter!r} true={true_letter!r}  raw_output={predicted_text!r}")

val_accuracy = correct / len(val_raw)
print(f"\nValidation accuracy: {correct}/{len(val_raw)} = {val_accuracy:.1%}")


correct predicted='A' true='A'  raw_output='A'
correct predicted='A' true='A'  raw_output='A'
correct predicted='A' true='A'  raw_output='A'
incorrect predicted='A' true='D'  raw_output='A'
correct predicted='B' true='B'  raw_output='B'
incorrect predicted='D' true='A'  raw_output='D'
correct predicted='A' true='A'  raw_output='A'
correct predicted='C' true='C'  raw_output='C'
correct predicted='D' true='D'  raw_output='D'
correct predicted='D' true='D'  raw_output='D'
correct predicted='A' true='A'  raw_output='A'
correct predicted='D' true='D'  raw_output='D'
incorrect predicted='A' true='C'  raw_output='A'
correct predicted='D' true='D'  raw_output='D'
correct predicted='D' true='D'  raw_output='D'
correct predicted='C' true='C'  raw_output='C'
correct predicted='D' true='D'  raw_output='D'
correct predicted='C' true='C'  raw_output='C'
incorrect predicted='D' true='B'  raw_output='D'
correct predicted='C' true='C'  raw_output='C'

Validation accuracy: 16/20 = 80.0%


### 10. Held-out test accuracy.
What does it actually get right, on data never touched?

Runs the fine-tuned model on test_raw (100 examples, never used in training, checkpoint selection, or early stopping).
- Unlike val_raw, this data had zero influence on any decision made during training (unbiased performance).

In [11]:
model.eval()

correct = 0
for example in test_raw:
    choices = ast.literal_eval(example["choices"])
    letters = [chr(ord("A") + i) for i in range(len(choices))]
    options_text = "\n".join(f"{letters[i]}. {choices[i]}" for i in range(len(choices)))
    true_letter = letters[example["correct_answer"]]

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": f"{example['question']}\n{options_text}"},
        ]},
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[[example["image"]]], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    predicted_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    predicted_letter = predicted_text[:1].upper()

    is_correct = predicted_letter == true_letter
    correct += is_correct
    print(f"{"correct" if is_correct else "incorrect"} predicted={predicted_letter!r} true={true_letter!r}  raw_output={predicted_text!r}")

test_accuracy = correct / len(test_raw)
print(f"\nTest accuracy: {correct}/{len(test_raw)} = {test_accuracy:.1%}")


incorrect predicted='C' true='D'  raw_output='C'
correct predicted='A' true='A'  raw_output='A'
correct predicted='C' true='C'  raw_output='C'
correct predicted='D' true='D'  raw_output='D'
incorrect predicted='D' true='B'  raw_output='D'
correct predicted='B' true='B'  raw_output='B'
incorrect predicted='E' true='A'  raw_output='E'
correct predicted='D' true='D'  raw_output='D'
correct predicted='B' true='B'  raw_output='B'
incorrect predicted='C' true='B'  raw_output='C'
correct predicted='C' true='C'  raw_output='C'
incorrect predicted='A' true='E'  raw_output='A'
correct predicted='D' true='D'  raw_output='D'
correct predicted='C' true='C'  raw_output='C'
correct predicted='A' true='A'  raw_output='A'
incorrect predicted='C' true='B'  raw_output='C'
correct predicted='A' true='A'  raw_output='A'
incorrect predicted='E' true='A'  raw_output='E'
incorrect predicted='A' true='E'  raw_output='A'
correct predicted='D' true='D'  raw_output='D'
correct predicted='D' true='D'  raw_output='